# Country-Year Panel of Age- and Sex-Specific Daily Water Turnover Estimates for Children and Adolescents (0–19 Years) Across 128 Countries, 1990–2022 Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.46w6-zpxe/fair2.json
```

This dataset contains harmonized, model-based daily water turnover estimates for children and adolescents aged 0 to 19 years from 128 countries between 1990 and 2022. Values are derived by combining isotope-validated water turnover equations with publicly available data on climate, elevation, human development index, fat-free mass, and physical activity, adjusted for national overweight/obesity and physical inactivity prevalence.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.46w6-zpxe/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets use the `@id` attribute to uniquely identify record sets, fields, and columns. Let's list them.

In [ ]:
# List all record sets, their fields, and columns by @id
record_sets = []
field_ids_per_record_set = {}

for rs in dataset.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs['@id']}")
    record_sets.append(rs['@id'])
    field_ids = []
    for field in rs.fields:
        print(f"  Field name: {getattr(field, 'name', None)}, @id: {field['@id']}, dataType: {getattr(field, 'dataType', None)}")
        field_ids.append(field['@id'])
        # If the field has columns, print those
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"    Column name: {getattr(col, 'name', None)}, @id: {col['@id']}, dataType: {getattr(col, 'dataType', None)}")
    field_ids_per_record_set[rs['@id']] = field_ids
print("\nAll RecordSets IDs:", record_sets)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecordSet {record_set_id} DataFrame shape: {df.shape}")
    print("Fields (columns):", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**All field and column references are by `@id`, as per Croissant standards.**

In [ ]:
# For demonstration, select the first record set and its first numeric field
selected_record_set_id = record_sets[0]
df = dataframes[selected_record_set_id]
fields_in_set = field_ids_per_record_set[selected_record_set_id]

# Try to find a numeric field (Float or Number) in this record set
numeric_field_id = None
for field in dataset.record_set(selected_record_set_id).fields:
    if getattr(field, 'dataType', '').lower() in ['float', 'number', 'integer']:
        numeric_field_id = field['@id']
        break

if numeric_field_id is None:
    print("No numeric field found in the selected record set.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")
    # Check if numeric_field_id actually appears as a column (it should)
    if numeric_field_id not in df.columns:
        print(f"Field {numeric_field_id} not found in DataFrame columns.")
    else:
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by another field, if possible
        group_field_id = None
        # Try to find a string (category) field
        for field in dataset.record_set(selected_record_set_id).fields:
            if getattr(field, 'dataType', '').lower() in ['string', 'text']:
                group_field_id = field['@id']
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No group field found or group field absent in dataframe.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

**Example: plot the distribution of selected numeric field**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if numeric_field_id is available and column exists
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Load a dataset defined by a Croissant schema using `mlcroissant`.
- Access metadata and enumerate record sets, fields, and their `@id`s for reproducible reference.
- Extract records as DataFrames and perform basic data analysis steps, including filtering and normalization.
- Visualize the distribution of numeric fields from the dataset.

The `FAIR^2` dataset is suitable for comparative epidemiological studies, modeling hydration needs worldwide, and ecological assessments linking water demand to climate and demographics. Ensure all field/column references use the Croissant `@id` for reproducibility and interoperability.